In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# download dataset
link = 'http://host.robots.ox.ac.uk/pascal/VOC/voc2008/VOCtrainval_14-Jul-2008.tar'
!wget $link

--2025-04-14 07:02:02--  http://host.robots.ox.ac.uk/pascal/VOC/voc2008/VOCtrainval_14-Jul-2008.tar
Resolving host.robots.ox.ac.uk (host.robots.ox.ac.uk)... 129.67.94.152
Connecting to host.robots.ox.ac.uk (host.robots.ox.ac.uk)|129.67.94.152|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 577034240 (550M) [application/x-tar]
Saving to: ‘VOCtrainval_14-Jul-2008.tar’

VOCtrainval_14-Jul- 100%[===================>] 550.30M  16.2MB/s    in 39s     

2025-04-14 07:02:41 (14.2 MB/s) - ‘VOCtrainval_14-Jul-2008.tar’ saved [577034240/577034240]



In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset, Subset
from torchvision.datasets import VOCDetection
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
import torchvision.utils as vutils

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os
import time
import gc
from collections import defaultdict
from tqdm.notebook import tqdm  # Use notebook tqdm for Kaggle
from sklearn.metrics import average_precision_score
# Mixed Precision
from torch.cuda.amp import GradScaler, autocast
import random

In [ ]:
CONFIG = {
    "data_dir": "./VOCdevkit/VOC2008", 
    "year": "2008",
    "image_set_train": "train",
    "image_set_val": "val",
    "classes": [
        'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat',
        'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person',
        'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
    ],
    "num_classes": 20,
    "gan_img_size": 64, # Smaller size for GAN training feasibility
    "classifier_img_size": 224, # Standard size for ResNet
    "gan_nz": 100,       # Size of z latent vector (i.e. size of generator input)
    "gan_ngf": 64,       # Size of feature maps in generator
    "gan_ndf": 64,       # Size of feature maps in discriminator
    "gan_epochs": 150,   # Increased from 50 to 150 for better convergence
    "gan_batch_size": 32, # Reduced from 64 for more stable gradients
    "gan_lr_g": 0.0001,  # Lower learning rate for generator (was 0.0002)
    "gan_lr_d": 0.00005, # Even lower learning rate for discriminator to prevent it from overwhelming the generator
    "gan_beta1": 0.5,    # Beta1 for Adam optimizer
    "gan_label_smoothing": 0.1, # Add label smoothing to help stabilize training
    "gan_early_stopping_patience": 20,  # Number of epochs for GAN early stopping
    "gan_early_stopping_threshold": 0.05,  # Relative improvement threshold for GAN
    "classifier_epochs": 25, # Adjust based on convergence & time
    "classifier_batch_size": 32, # Adjust based on GPU memory
    "classifier_lr": 0.001,
    "early_stopping_patience": 5,  # Number of epochs with no improvement to wait before stopping
    "early_stopping_min_delta": 0.001,  # Minimum change to qualify as improvement
    "augmentation_samples": [0, 100, 200, 500], # 0 is baseline (no augmentation | original samples)
    "results_dir": "./results",
    "gan_models_dir": "./gan_models",
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "seed": 42,
}

# Create directories
os.makedirs(CONFIG["results_dir"], exist_ok=True)
os.makedirs(CONFIG["gan_models_dir"], exist_ok=True)

In [ ]:
# Utility function to download and extract VOC dataset
def download_voc_dataset():
    """Download and extract VOC2008 dataset if it doesn't exist."""
    import urllib.request
    import tarfile
    
    voc_url = "http://host.robots.ox.ac.uk/pascal/VOC/voc2008/VOCtrainval_14-Jul-2008.tar"
    tar_file_path = "./VOCtrainval_14-Jul-2008.tar"
    
    # Download if the tar file doesn't exist
    if not os.path.exists(tar_file_path):
        print(f"Downloading VOC2008 dataset from {voc_url}...")
        try:
            urllib.request.urlretrieve(voc_url, tar_file_path)
            print("Download complete.")
        except Exception as e:
            print(f"Error downloading dataset: {e}")
            print(f"Please download manually from {voc_url}")
            return False
    else:
        print(f"Found existing VOC dataset file: {tar_file_path}")
    
    # Extract the tar file
    try:
        print("Extracting dataset...")
        with tarfile.open(tar_file_path) as tar:
            tar.extractall(path=".")
        print("Extraction complete.")
        return True
    except Exception as e:
        print(f"Error extracting dataset: {e}")
        return False

In [ ]:
# Check if VOC dataset structure exists
voc_data_dir = CONFIG["data_dir"]
voc_jpeg_dir = os.path.join(voc_data_dir, 'JPEGImages')
voc_imageset_dir = os.path.join(voc_data_dir, 'ImageSets', 'Main')

# If dataset structure doesn't exist, try to download it
if not os.path.exists(voc_data_dir):
    print(f"VOC dataset directory not found: {voc_data_dir}")
    if download_voc_dataset():
        print("VOC dataset has been downloaded and extracted.")
    else:
        print("Could not download VOC dataset automatically.")
        print("Please download and extract it manually to continue.")
        print("Expected structure: ./VOCdevkit/VOC2008/")

# Set seed for reproducibility
torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])
    # Enable benchmark mode for potential performance improvements if input sizes don't vary
    torch.backends.cudnn.benchmark = True

print(f"Using device: {CONFIG['device']}")
print(f"Number of classes: {CONFIG['num_classes']}")
print(f"Class names: {CONFIG['classes']}")

In [ ]:
class VOCClassificationCustomDataset(Dataset):
    def __init__(self, base_dir, image_set, transform=None, return_img_path=False):
        # Check and adjust image directory name if needed
        self.image_dir = os.path.join(base_dir, 'JPEGImages')
        if not os.path.exists(self.image_dir):
            alternative_dirs = ['images', 'Images', 'JPEG', 'jpeg']
            for alt_dir in alternative_dirs:
                alt_path = os.path.join(base_dir, alt_dir)
                if os.path.exists(alt_path):
                    self.image_dir = alt_path
                    print(f"Using alternative image directory: {alt_path}")
                    break
        
        self.imageset_dir = os.path.join(base_dir, 'ImageSets', 'Main')
        if not os.path.exists(self.imageset_dir):
            alt_path = os.path.join(base_dir, 'ImageSets')
            if os.path.exists(alt_path):
                self.imageset_dir = alt_path
                print(f"Using alternative imageset directory: {alt_path}")
        
        self.transform = transform
        self.image_set = image_set
        self.return_img_path = return_img_path # Flag to return image path for visualization
        self.classes = CONFIG['classes']
        self.num_classes = CONFIG['num_classes']
        
        # Check if image_set.txt exists (train.txt or val.txt)
        split_file = os.path.join(self.imageset_dir, f"{image_set}.txt")
        
        # If it doesn't exist, try trainval.txt instead
        if not os.path.exists(split_file):
            print(f"Warning: {split_file} not found.")
            trainval_file = os.path.join(self.imageset_dir, "trainval.txt")
            
            if os.path.exists(trainval_file):
                print(f"Using trainval.txt instead for {image_set}")
                split_file = trainval_file
            else:
                # If trainval.txt also doesn't exist, create image_set.txt from class-specific files
                print(f"Creating {image_set}.txt from class-specific files...")
                self._create_image_set_file(image_set)
                
                # Check if we successfully created the file
                if not os.path.exists(split_file):
                    raise FileNotFoundError(f"Could not create {split_file}. Check that class-specific files exist.")
        
        with open(split_file, 'r') as f: 
            self.image_ids = [line.strip() for line in f if line.strip()]
        
        self._load_labels()
        print(f"Found {len(self.image_ids)} IDs, loaded labels for Cls '{self.image_set}' set.")
        if not self.image_ids: 
            print(f"Warning: Cls image set '{image_set}' is empty.")
    
    def _create_image_set_file(self, image_set):
        """Create image_set.txt file from class-specific files"""
        # First, collect all image IDs from class-specific files
        all_image_ids = set()
        
        for class_name in self.classes:
            class_file = os.path.join(self.imageset_dir, f"{class_name}_{image_set}.txt")
            
            # If we can't find class_name_image_set.txt, try class_name_trainval.txt
            if not os.path.exists(class_file):
                class_file = os.path.join(self.imageset_dir, f"{class_name}_trainval.txt")
                
            if os.path.exists(class_file):
                with open(class_file, 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) >= 1:
                            img_id = parts[0]
                            all_image_ids.add(img_id)
        
        # Write image IDs to image_set.txt
        if all_image_ids:
            output_file = os.path.join(self.imageset_dir, f"{image_set}.txt")
            with open(output_file, 'w') as f:
                for img_id in sorted(all_image_ids):
                    f.write(f"{img_id}\n")
            print(f"Created {output_file} with {len(all_image_ids)} image IDs.")

    def _load_labels(self):
        self.labels = {}
        class_label_data = {}
        
        for class_name in self.classes:
            # Try to find class file with format: class_image_set.txt
            class_file = os.path.join(self.imageset_dir, f"{class_name}_{self.image_set}.txt")
            
            # If it doesn't exist, try trainval
            if not os.path.exists(class_file):
                class_file = os.path.join(self.imageset_dir, f"{class_name}_trainval.txt")
            
            # Initialize mapping for this class
            image_to_label = {}
            class_label_data[class_name] = image_to_label
            
            if os.path.exists(class_file):
                with open(class_file, 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) == 2: 
                            img_id, label_val = parts[0], int(parts[1])
                            image_to_label[img_id] = label_val
                print(f"Loaded {len(image_to_label)} labels for class '{class_name}'")
            else:
                print(f"Warning: No label file found for class '{class_name}'")
                        
        for img_id in self.image_ids:
            label_vector = torch.zeros(self.num_classes, dtype=torch.float32)
            for i, class_name in enumerate(self.classes):
                if class_label_data[class_name].get(img_id, 0) == 1: 
                    label_vector[i] = 1.0
            self.labels[img_id] = label_vector

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_path = os.path.join(self.image_dir, f"{img_id}.jpg")
        label = self.labels.get(img_id, torch.zeros(self.num_classes, dtype=torch.float32))
        dummy_image = Image.new('RGB', (224, 224), color='grey')
        
        # Try different image extensions if .jpg doesn't exist
        if not os.path.exists(img_path):
            for ext in ['.png', '.jpeg', '.JPG', '.JPEG', '.PNG']:
                alt_path = os.path.join(self.image_dir, f"{img_id}{ext}")
                if os.path.exists(alt_path):
                    img_path = alt_path
                    break
        
        try:
            image = Image.open(img_path).convert('RGB')
        except (FileNotFoundError, IOError) as e: 
            print(f"\nError: Cannot load image file: {img_path} - {e}")
            image = dummy_image

        if self.transform:
            try: 
                transformed_image = self.transform(image)
            except Exception as e:
                print(f"\nError transforming image {img_id}: {e}")
                transformed_image = self.transform(dummy_image) if self.transform else transforms.ToTensor()(dummy_image)
        else:
             transformed_image = transforms.ToTensor()(image) # Basic transform if none provided

        if self.return_img_path:
             return transformed_image, label, img_path # Return path if requested
        else:
             return transformed_image, label

In [ ]:
# For GAN Training (smaller size, simple normalization)
gan_transform = transforms.Compose([
    transforms.Resize(CONFIG['gan_img_size']),
    transforms.CenterCrop(CONFIG['gan_img_size']),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

# For Classifier Training/Evaluation (larger size, ImageNet normalization)
classifier_train_transform = transforms.Compose([
    transforms.Resize((CONFIG['classifier_img_size'], CONFIG['classifier_img_size'])),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),  # Add color jittering for better augmentation
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

classifier_val_transform = transforms.Compose([
    transforms.Resize((CONFIG['classifier_img_size'], CONFIG['classifier_img_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
# --- IMPORTANT: Create separate dataset instances for GAN and Classifier as they use different transforms ---

# Original Training Data (for classifier baseline & augmentation source)
original_train_dataset_classifier = VOCClassificationCustomDataset(
    base_dir=CONFIG['data_dir'],
    image_set=CONFIG['image_set_train'],
    transform=classifier_train_transform
)

# Validation Data (for classifier evaluation
val_dataset_classifier = VOCClassificationCustomDataset(
    base_dir=CONFIG['data_dir'],
    image_set=CONFIG['image_set_val'],
    transform=classifier_val_transform
)

val_loader = DataLoader(val_dataset_classifier, batch_size=CONFIG['classifier_batch_size'], shuffle=False, num_workers=2, pin_memory=True)

print(f"Original training samples (for classifier): {len(original_train_dataset_classifier)}")
print(f"Validation samples (for classifier): {len(val_dataset_classifier)}")


In [ ]:
# --- 2. GAN Implementation (DCGAN) ---

# Custom weights initialization called on netG and netD
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

# Generator Code
class Generator(nn.Module):
    def __init__(self, nz, ngf, nc=3):
        super(Generator, self).__init__()
        self.main = nn.Sequential(
            # input is Z, going into a convolution
            nn.ConvTranspose2d( nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            # state size. (ngf*8) x 4 x 4
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # state size. (ngf*4) x 8 x 8
            nn.ConvTranspose2d( ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # state size. (ngf*2) x 16 x 16
            nn.ConvTranspose2d( ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # state size. (ngf) x 32 x 32
            nn.ConvTranspose2d( ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh()
            # state size. (nc) x 64 x 64
        )

    def forward(self, input):
        return self.main(input)

# Discriminator Code
class Discriminator(nn.Module):
    def __init__(self, ndf, nc=3):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            # input is (nc) x 64 x 64
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(0.2),  # Add dropout for regularization
            # state size. (ndf) x 32 x 32
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(0.3),  # Add dropout for regularization
            # state size. (ndf*2) x 16 x 16
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(0.3),  # Add dropout for regularization
            # state size. (ndf*4) x 8 x 8
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(0.2),  # Add dropout for regularization
            # state size. (ndf*8) x 4 x 4
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, input):
        return self.main(input)

In [ ]:
# --- GAN Training Function ---
def train_gan_for_class(class_name, class_idx, data_dir, image_set, gan_transform, config):
    print(f"\n--- Training GAN for class: {class_name} ---")
    gan_model_path = os.path.join(config['gan_models_dir'], f"generator_{class_name}.pth")
    if os.path.exists(gan_model_path):
        print(f"GAN generator model already exists for {class_name}. Skipping training.")
        return

    # --- Create a dataset containing ONLY images with the target class ---
    # Use a dedicated dataset instance with GAN transforms
    gan_train_dataset_full = VOCClassificationCustomDataset(
        base_dir=data_dir,
        image_set=image_set,
        transform=gan_transform
    )

    # Filter indices for the current class
    indices = []
    for i, img_id in enumerate(gan_train_dataset_full.image_ids):
        label = gan_train_dataset_full.labels.get(img_id)
        if label is not None and label[class_idx] == 1:
            indices.append(i)

    if not indices:
        print(f"Warning: No training images found for class {class_name}. Cannot train GAN.")
        return

    print(f"Found {len(indices)} training images containing class {class_name}.")
    class_dataset = Subset(gan_train_dataset_full, indices)
    dataloader = DataLoader(class_dataset, batch_size=config['gan_batch_size'], shuffle=True, num_workers=2, pin_memory=True, drop_last=True)

    # Initialize models
    netG = Generator(config['gan_nz'], config['gan_ngf']).to(config['device'])
    netD = Discriminator(config['gan_ndf']).to(config['device'])
    netG.apply(weights_init)
    netD.apply(weights_init)

    # Loss and Optimizers
    criterion = nn.BCELoss()
    optimizerD = optim.Adam(netD.parameters(), lr=config['gan_lr_d'], betas=(config['gan_beta1'], 0.999))
    optimizerG = optim.Adam(netG.parameters(), lr=config['gan_lr_g'], betas=(config['gan_beta1'], 0.999))
    
    # Learning rate schedulers - reduce learning rate over time
    schedulerD = optim.lr_scheduler.StepLR(optimizerD, step_size=30, gamma=0.5)
    schedulerG = optim.lr_scheduler.StepLR(optimizerG, step_size=30, gamma=0.5)

    # Fixed noise for visualization
    fixed_noise = torch.randn(config['gan_batch_size'], config['gan_nz'], 1, 1, device=config['device'])

    # Establish convention for real and fake labels during training
    real_label = 1.
    fake_label = 0.
    
    # Instance noise - decrease over time
    noise_std = 0.1  # Initial standard deviation of noise

    # Training Loop
    img_list = []
    G_losses = []
    D_losses = []
    iters = 0
    start_time = time.time()
    
    # Early stopping variables
    patience = config.get('gan_early_stopping_patience', 20)
    min_improvement = config.get('gan_early_stopping_threshold', 0.05)
    best_g_loss = float('inf')
    best_epoch = 0
    stop_training = False
    window_size = 10  # Window size for loss smoothing

    print("Starting GAN Training Loop...")
    for epoch in range(config['gan_epochs']):
        if stop_training:
            print(f"Early stopping GAN training at epoch {epoch}")
            break
            
        epoch_start_time = time.time()
        # Decrease noise standard deviation over epochs
        current_noise_std = max(0, noise_std * (1 - epoch/config['gan_epochs']))
        
        # Track losses for this epoch
        epoch_g_losses = []
        
        for i, data in enumerate(dataloader, 0):
            # --- Train Discriminator ---
            netD.zero_grad()
            # Format batch
            real_cpu = data[0].to(config['device'])
            b_size = real_cpu.size(0)
            
            # Add instance noise to real images (decreasing over time)
            if current_noise_std > 0:
                real_cpu = real_cpu + current_noise_std * torch.randn_like(real_cpu)
                # Clamp values to valid range [-1, 1]
                real_cpu = torch.clamp(real_cpu, -1, 1)
            
            # Label smoothing - use soft labels for real images
            smooth_real_labels = torch.full((b_size,), real_label - config['gan_label_smoothing'], 
                                          dtype=torch.float, device=config['device'])
            
            # Forward pass real batch through D
            output = netD(real_cpu).view(-1)
            # Calculate loss on all-real batch
            errD_real = criterion(output, smooth_real_labels)
            # Calculate gradients for D in backward pass
            errD_real.backward()
            D_x = output.mean().item()

            ## Train with all-fake batch
            # Generate batch of latent vectors
            noise = torch.randn(b_size, config['gan_nz'], 1, 1, device=config['device'])
            # Generate fake image batch with G
            fake = netG(noise)
            
            # Add instance noise to fake images (decreasing over time)
            if current_noise_std > 0:
                fake = fake + current_noise_std * torch.randn_like(fake)
                # Clamp values to valid range [-1, 1]
                fake = torch.clamp(fake, -1, 1)
                
            label = torch.full((b_size,), fake_label, dtype=torch.float, device=config['device'])
            # Classify all fake batch with D
            output = netD(fake.detach()).view(-1) # detach G's history
            # Calculate D's loss on the all-fake batch
            errD_fake = criterion(output, label)
            # Calculate the gradients for this batch
            errD_fake.backward()
            D_G_z1 = output.mean().item()
            # Add the gradients from the all-real and all-fake batches
            errD = errD_real + errD_fake
            # Update D
            optimizerD.step()

            # --- Train Generator ---
            netG.zero_grad()
            # We want to use soft labels here too (a bit less than 1.0) for stability
            soft_labels = torch.full((b_size,), real_label - config['gan_label_smoothing']/2, 
                                    dtype=torch.float, device=config['device'])
            # Since we just updated D, perform another forward pass of all-fake batch through D
            output = netD(fake).view(-1)
            # Calculate G's loss based on this output
            errG = criterion(output, soft_labels)
            # Calculate gradients for G
            errG.backward()
            D_G_z2 = output.mean().item()
            # Update G
            optimizerG.step()
            
            # Save generator loss for early stopping
            epoch_g_losses.append(errG.item())

            # Output training stats
            if i % 50 == 0:
                print(f'[{epoch+1}/{config["gan_epochs"]}][{i}/{len(dataloader)}] Loss_D: {errD.item():.4f} Loss_G: {errG.item():.4f} D(x): {D_x:.4f} D(G(z)): {D_G_z1:.4f} / {D_G_z2:.4f} Noise: {current_noise_std:.4f}')

            # Save Losses for plotting later
            G_losses.append(errG.item())
            D_losses.append(errD.item())

            # Check how the generator is doing by saving G's output on fixed_noise
            if (iters % 200 == 0) or ((epoch == config['gan_epochs']-1) and (i == len(dataloader)-1)):
                 with torch.no_grad():
                    fake = netG(fixed_noise).detach().cpu()
                 img_list.append(vutils.make_grid(fake, padding=2, normalize=True))
                 # Save grid image
                 grid_filename = os.path.join(config['results_dir'], f"gan_progress_{class_name}_epoch{epoch+1}.png")
                 vutils.save_image(img_list[-1], grid_filename)

            iters += 1
            
        # Calculate average G loss for this epoch
        avg_g_loss = sum(epoch_g_losses) / len(epoch_g_losses) if epoch_g_losses else float('inf')
        
        # Early stopping check - smooth loss over window if possible
        if len(G_losses) >= window_size:
            # Use a moving average of G_losses for smoother early stopping
            recent_g_losses = G_losses[-window_size:]
            smoothed_g_loss = sum(recent_g_losses) / window_size
        else:
            smoothed_g_loss = avg_g_loss
            
        # Check for improvement
        relative_improvement = (best_g_loss - smoothed_g_loss) / best_g_loss if best_g_loss > 0 else 1.0
        
        if smoothed_g_loss < best_g_loss or relative_improvement > min_improvement:
            best_g_loss = smoothed_g_loss
            best_epoch = epoch
            # Save the current best model
            torch.save(netG.state_dict(), gan_model_path)
            print(f"New best G loss: {best_g_loss:.4f} (saved model)")
        
        # Check for early stopping
        epochs_without_improvement = epoch - best_epoch
        if epoch > 30 and epochs_without_improvement >= patience:  # Use early stopping only after 30 epochs
            print(f"Early stopping triggered. No improvement for {epochs_without_improvement} epochs.")
            stop_training = True
        
        epoch_time = time.time() - epoch_start_time
        print(f"Epoch {epoch+1} finished in {epoch_time:.2f} seconds. G Loss: {avg_g_loss:.4f}, Best: {best_g_loss:.4f}, Epochs without improvement: {epochs_without_improvement}")
        
        # Step the learning rate schedulers
        schedulerD.step()
        schedulerG.step()
        current_lr_g = optimizerG.param_groups[0]['lr']
        current_lr_d = optimizerD.param_groups[0]['lr']
        print(f"Learning rates: G={current_lr_g:.6f}, D={current_lr_d:.6f}")

    total_time = time.time() - start_time
    print(f"Finished GAN training for {class_name} in {total_time:.2f} seconds.")

    # Plot losses
    plt.figure(figsize=(10,5))
    plt.title(f"Generator and Discriminator Loss During Training ({class_name})")
    plt.plot(G_losses,label="G")
    plt.plot(D_losses,label="D")
    plt.xlabel("iterations")
    plt.ylabel("Loss")
    plt.legend()
    plt.savefig(os.path.join(config['results_dir'], f"gan_loss_{class_name}.png"))
    # plt.show() # Avoid showing in Kaggle script mode
    plt.close() # Close plot to free memory

    # Clean up
    del netG, netD, optimizerG, optimizerD, dataloader, class_dataset, gan_train_dataset_full
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
def generate_synthetic_data(n_samples_per_class, config):
    synthetic_images = []
    synthetic_labels = []
    generator = Generator(config['gan_nz'], config['gan_ngf']).to(config['device'])

    # Define the inverse normalization transform for visualization/saving if needed
    # inv_normalize = transforms.Normalize(
    #    mean=[-0.5/0.5, -0.5/0.5, -0.5/0.5],
    #    std=[1/0.5, 1/0.5, 1/0.5]
    #)
    # Define transform to resize GAN output to Classifier input size and normalize
    resize_normalize_transform = transforms.Compose([
        transforms.Resize((config['classifier_img_size'], config['classifier_img_size'])),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # Use ImageNet norm
    ])


    print(f"\n--- Generating {n_samples_per_class} synthetic samples per class ---")
    generated_samples_count = 0  # Track total generated samples
    for class_idx, class_name in enumerate(tqdm(config['classes'])):
        gan_model_path = os.path.join(config['gan_models_dir'], f"generator_{class_name}.pth")
        if not os.path.exists(gan_model_path):
            print(f"Warning: Generator model not found for {class_name}. Skipping generation for this class.")
            continue

        generator.load_state_dict(torch.load(gan_model_path, map_location=config['device']))
        generator.eval()
        
        # Use a slightly larger batch of noise and select best samples
        # This helps filter out poor quality generations
        selection_factor = 2  # Generate 2x samples and select best ones
        generation_batch_size = min(64, n_samples_per_class * selection_factor)
        
        generated_count = 0
        with torch.no_grad():
            while generated_count < n_samples_per_class:
                batch_size = min(generation_batch_size, n_samples_per_class - generated_count)
                if batch_size <= 0: break

                # Generate more samples than needed and select the best ones
                noise = torch.randn(batch_size * selection_factor, config['gan_nz'], 1, 1, device=config['device'])
                # Generate images in [-1, 1] range (due to Tanh)
                gen_imgs_gan_norm = generator(noise)

                # --- IMPORTANT: Post-process generated images ---
                # 1. Denormalize from GAN's [-1, 1] to [0, 1]
                gen_imgs_0_1 = gen_imgs_gan_norm * 0.5 + 0.5
                
                # Calculate image quality scores (based on standard deviation - higher is better)
                # Images with very low variance are often mode-collapsed and poor quality
                quality_scores = []
                for img in gen_imgs_0_1:
                    std_score = torch.std(img).item()  # Standard deviation - higher is better
                    quality_scores.append(std_score)
                
                # Select top quality images
                indices = sorted(range(len(quality_scores)), key=lambda i: quality_scores[i], reverse=True)
                selected_indices = indices[:batch_size]
                selected_imgs = gen_imgs_0_1[selected_indices]
                
                # 2. Resize to classifier input size and apply classifier normalization
                # Need to apply transform to each image in the batch
                processed_imgs = torch.stack([resize_normalize_transform(img) for img in selected_imgs.cpu()]).to(config['device'])

                # Create labels (one-hot for the generated class)
                # NOTE: This is a simplification. Real images are multi-label.
                # GAN generated images are assigned only the label of the class they were trained for.
                label_vector = torch.zeros(config['num_classes'], device=config['device'])
                label_vector[class_idx] = 1.0
                batch_labels = label_vector.repeat(batch_size, 1)

                synthetic_images.append(processed_imgs.cpu()) # Move to CPU to save GPU RAM
                synthetic_labels.append(batch_labels.cpu())
                
                generated_count += batch_size
                generated_samples_count += batch_size

    # Clean up generator from GPU memory
    del generator
    gc.collect()
    torch.cuda.empty_cache()

    if not synthetic_images:
        return None, None

    all_synthetic_images = torch.cat(synthetic_images, dim=0)
    all_synthetic_labels = torch.cat(synthetic_labels, dim=0)
    print(f"Generated {len(all_synthetic_images)} total synthetic samples across {len(config['classes'])} classes.")
    return all_synthetic_images, all_synthetic_labels


In [ ]:
class AugmentedDataset(Dataset):
    def __init__(self, original_dataset, synthetic_images, synthetic_labels, synthetic_weight=0.5):
        """
        Create an augmented dataset that combines original and synthetic data.
        
        Args:
            original_dataset: Dataset containing original training data
            synthetic_images: Tensor of synthetic images generated by GAN
            synthetic_labels: Tensor of labels for synthetic images
            synthetic_weight: Weight to give synthetic samples (0.0-1.0)
        """
        self.original_dataset = original_dataset
        self.synthetic_images = synthetic_images
        self.synthetic_labels = synthetic_labels
        self.synthetic_weight = min(1.0, max(0.0, synthetic_weight))  # Clamp between 0 and 1

        if synthetic_images is None or synthetic_labels is None:
            self.synthetic_len = 0
        else:
            self.synthetic_len = len(synthetic_images)

        self.original_len = len(original_dataset)
        self.total_len = self.original_len + self.synthetic_len
        
        # Create class counts for both datasets
        self.analyze_class_distribution()
        
        print(f"Created augmented dataset with {self.original_len} original and {self.synthetic_len} synthetic samples")
        print(f"Using synthetic weight of {self.synthetic_weight:.2f}")
    
    def analyze_class_distribution(self):
        """Analyze and report class distribution in both datasets"""
        if self.synthetic_len == 0:
            return
            
        # Count classes in original dataset
        orig_class_counts = torch.zeros(CONFIG['num_classes'])
        for i in range(self.original_len):
            _, label = self.original_dataset[i]
            orig_class_counts += label
            
        # Count classes in synthetic dataset
        synth_class_counts = torch.sum(self.synthetic_labels, dim=0)
        
        # Print class distribution summary
        print("\nClass distribution analysis:")
        print(f"{'Class':<15} {'Original':<10} {'Synthetic':<10} {'Ratio':<10}")
        print("-" * 45)
        
        for i, class_name in enumerate(CONFIG['classes']):
            orig = orig_class_counts[i].item()
            synth = synth_class_counts[i].item()
            ratio = synth / max(1, orig)  # Avoid division by zero
            print(f"{class_name:<15} {int(orig):<10} {int(synth):<10} {ratio:.2f}")

    def __len__(self):
        return self.total_len

    def __getitem__(self, idx):
        # Determine whether to sample from original or synthetic based on synthetic_weight
        # and the relative sizes of the datasets
        p_synthetic = self.synthetic_weight * self.synthetic_len / self.total_len
        
        if idx < self.original_len and (self.synthetic_len == 0 or random.random() > p_synthetic):
            # Get from original dataset
            return self.original_dataset[idx]
        else:
            # Get from synthetic data
            # For synthetic data, randomly sample rather than using a fixed index
            # This prevents bias when using non-uniform synthetic data quantities
            synthetic_idx = random.randint(0, self.synthetic_len - 1)
            img = self.synthetic_images[synthetic_idx]
            label = self.synthetic_labels[synthetic_idx]
            return img, label

In [ ]:
def get_classifier(num_classes, pretrained=True):
    if pretrained:
        weights = ResNet18_Weights.IMAGENET1K_V1
        model = resnet18(weights=weights)
    else:
        model = resnet18(weights=None)

    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes) # Replace the final layer
    return model

# --- Classifier Training Function ---
def train_classifier(model, train_loader, val_loader, config):
    model.to(config['device'])
    # Use BCEWithLogitsLoss for multi-label classification
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=config['classifier_lr'])
    # Learning rate scheduler (optional but often helpful)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)
    # GradScaler for Mixed Precision
    scaler = GradScaler(enabled=torch.cuda.is_available())

    best_map = 0.0
    best_model_state = None
    
    # Early stopping variables
    patience = config.get('early_stopping_patience', 5)
    min_delta = config.get('early_stopping_min_delta', 0.001)
    counter = 0
    early_stop = False

    print("\n--- Starting Classifier Training ---")
    for epoch in range(config['classifier_epochs']):
        if early_stop:
            print(f"Early stopping triggered after {epoch} epochs!")
            break
            
        model.train()
        running_loss = 0.0
        train_start_time = time.time()

        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['classifier_epochs']} [Train]")
        for i, (inputs, labels) in enumerate(progress_bar):
            inputs, labels = inputs.to(config['device']), labels.to(config['device'])

            optimizer.zero_grad()

            # Mixed Precision Context
            with autocast(enabled=torch.cuda.is_available()):
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            # Scales loss. Calls backward() on scaled loss to create scaled gradients.
            scaler.scale(loss).backward()

            # scaler.step() first unscales the gradients of the optimizer's assigned params.
            # If these gradients do not contain infs or NaNs, optimizer.step() is then called.
            # Otherwise, optimizer.step() is skipped.
            scaler.step(optimizer)

            # Updates the scale for next iteration.
            scaler.update()

            running_loss += loss.item() * inputs.size(0)
            progress_bar.set_postfix(loss=f"{loss.item():.4f}")


        epoch_loss = running_loss / len(train_loader.dataset)
        train_time = time.time() - train_start_time

        # Validation phase (using evaluate_classifier for mAP)
        val_map, val_loss = evaluate_classifier(model, val_loader, criterion, config)

        print(f"Epoch {epoch+1}/{config['classifier_epochs']} - "
              f"Train Loss: {epoch_loss:.4f}, "
              f"Val Loss: {val_loss:.4f}, "
              f"Val mAP: {val_map:.4f}, "
              f"Train Time: {train_time:.2f}s")

        scheduler.step() # Step the scheduler

        # Save the best model based on validation mAP
        if val_map > best_map + min_delta:
            best_map = val_map
            # Save model state on CPU to avoid GPU memory issues when loading later
            best_model_state = {k: v.cpu() for k, v in model.state_dict().items()}
            print(f"*** New best model saved with mAP: {best_map:.4f} ***")
            counter = 0  # Reset early stopping counter
        else:
            counter += 1
            print(f"EarlyStopping counter: {counter} out of {patience}")
            if counter >= patience:
                early_stop = True
                print("Early stopping: Validation mAP did not improve for", patience, "epochs.")

    print("Finished Classifier Training.")
    # Load the best model state back
    if best_model_state:
        model.load_state_dict(best_model_state)
        print(f"Loaded best model state with mAP: {best_map:.4f}")
    else:
         print("Warning: No best model state was saved (validation mAP might not have improved).")

    return model, best_map # Return trained model and best mAP

In [ ]:
# --- Evaluation Function (Calculate mAP) ---
def calculate_ap(y_true, y_scores):
    """Calculate Average Precision (AP) for a single class."""
    # Ensure inputs are numpy arrays
    y_true = np.array(y_true)
    y_scores = np.array(y_scores)

    # Handle cases where a class is not present in the ground truth
    if np.sum(y_true) == 0:
        # print("Warning: No positive samples found for this class in validation set. Returning AP=0.")
        return 0.0 # Or np.nan, depending on how you want to handle this

    return average_precision_score(y_true, y_scores)

def calculate_map(all_labels, all_preds):
    """Calculate Mean Average Precision (mAP) across all classes."""
    num_classes = all_labels.shape[1]
    average_precisions = []
    for i in range(num_classes):
        ap = calculate_ap(all_labels[:, i], all_preds[:, i])
        average_precisions.append(ap)

    # Filter out potential NaN values if any class had no positive samples
    valid_aps = [ap for ap in average_precisions if not np.isnan(ap)]
    if not valid_aps:
        return 0.0 # Or np.nan
    return np.mean(valid_aps)


def evaluate_classifier(model, dataloader, criterion, config):
    model.to(config['device'])
    model.eval()
    all_labels = []
    all_preds = []
    running_loss = 0.0

    print("--- Evaluating Classifier ---")
    progress_bar = tqdm(dataloader, desc="Evaluation")
    with torch.no_grad():
        for inputs, labels in progress_bar:
            inputs, labels = inputs.to(config['device']), labels.to(config['device'])

            # Mixed precision inference
            with autocast(enabled=torch.cuda.is_available()):
                outputs = model(inputs)
                loss = criterion(outputs, labels) # Calculate loss if needed

            running_loss += loss.item() * inputs.size(0)

            # Store predictions (probabilities using sigmoid) and true labels
            # Move to CPU before converting to numpy to avoid GPU sync issues
            preds = torch.sigmoid(outputs).cpu().numpy()
            labels_np = labels.cpu().numpy()

            all_preds.append(preds)
            all_labels.append(labels_np)

    # Concatenate all batch results
    all_labels = np.concatenate(all_labels, axis=0)
    all_preds = np.concatenate(all_preds, axis=0)

    # Calculate mAP
    mean_ap = calculate_map(all_labels, all_preds)
    epoch_loss = running_loss / len(dataloader.dataset)

    return mean_ap, epoch_loss

# --- Save Sample Images from Each Class's GAN ---

def save_gan_samples(config, num_samples=5):
    """
    Save sample images generated by each class's GAN generator.
    
    Args:
        config: Configuration dictionary
        num_samples: Number of samples to save per class
    """
    print("\n--- Saving GAN Sample Images ---")
    
    # Create a directory for GAN samples
    samples_dir = os.path.join(config['results_dir'], "gan_samples")
    os.makedirs(samples_dir, exist_ok=True)
    
    # Initialize generator
    generator = Generator(config['gan_nz'], config['gan_ngf']).to(config['device'])
    
    # Define inverse normalization for visualization
    inv_normalize = transforms.Normalize(
        mean=[-0.5/0.5, -0.5/0.5, -0.5/0.5],
        std=[1/0.5, 1/0.5, 1/0.5]
    )
    
    for class_idx, class_name in enumerate(config['classes']):
        gan_model_path = os.path.join(config['gan_models_dir'], f"generator_{class_name}.pth")
        if not os.path.exists(gan_model_path):
            print(f"Warning: Generator model not found for {class_name}. Skipping sample generation.")
            continue
            
        print(f"Generating samples for class: {class_name}")
        
        # Load the trained generator
        generator.load_state_dict(torch.load(gan_model_path, map_location=config['device']))
        generator.eval()
        
        # Generate samples
        with torch.no_grad():
            # Generate a batch of images
            noise = torch.randn(num_samples, config['gan_nz'], 1, 1, device=config['device'])
            fake_images = generator(noise)
            
            # Denormalize for visualization
            fake_images = inv_normalize(fake_images)
            
            # Save individual images
            for i in range(num_samples):
                img_path = os.path.join(samples_dir, f"{class_name}_sample_{i+1}.png")
                vutils.save_image(fake_images[i], img_path)
            
            # Save a grid of all samples for this class
            grid_path = os.path.join(samples_dir, f"{class_name}_grid.png")
            grid = vutils.make_grid(fake_images, nrow=num_samples, padding=2, normalize=True)
            vutils.save_image(grid, grid_path)
    
    print(f"GAN samples saved to {samples_dir}")
    del generator
    gc.collect()
    torch.cuda.empty_cache()

def save_synthetic_samples(synthetic_images, synthetic_labels, config, num_samples=5):
    """
    Save sample images from the synthetic data generation process.
    
    Args:
        synthetic_images: Tensor of synthetic images
        synthetic_labels: Tensor of synthetic labels
        config: Configuration dictionary
        num_samples: Number of samples to save per class
    """
    print("\n--- Saving Synthetic Sample Images ---")
    
    # Create a directory for synthetic samples
    samples_dir = os.path.join(config['results_dir'], "synthetic_samples")
    os.makedirs(samples_dir, exist_ok=True)
    
    # Define inverse normalization for visualization
    inv_normalize = transforms.Normalize(
        mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
        std=[1/0.229, 1/0.224, 1/0.225]
    )
    
    # Convert labels to numpy for easier handling
    labels_np = synthetic_labels.numpy()
    
    # For each class, save a few samples
    for class_idx, class_name in enumerate(config['classes']):
        # Find indices of images with this class
        class_indices = np.where(labels_np[:, class_idx] == 1)[0]
        
        if len(class_indices) == 0:
            print(f"No synthetic samples found for class {class_name}")
            continue
            
        # Take a random subset if we have more than num_samples
        if len(class_indices) > num_samples:
            selected_indices = np.random.choice(class_indices, num_samples, replace=False)
        else:
            selected_indices = class_indices
            
        # Get the selected images
        selected_images = synthetic_images[selected_indices]
        
        # Denormalize for visualization
        denormalized_images = inv_normalize(selected_images)
        
        # Save individual images
        for i, img_idx in enumerate(selected_indices):
            img_path = os.path.join(samples_dir, f"{class_name}_synthetic_{i+1}.png")
            vutils.save_image(denormalized_images[i], img_path)
        
        # Save a grid of all samples for this class
        grid_path = os.path.join(samples_dir, f"{class_name}_synthetic_grid.png")
        grid = vutils.make_grid(denormalized_images, nrow=min(num_samples, len(selected_indices)), padding=2, normalize=True)
        vutils.save_image(grid, grid_path)
    
    print(f"Synthetic samples saved to {samples_dir}")

def save_original_samples(dataset, config, num_samples=5):
    """
    Save sample images from the original training dataset.
    
    Args:
        dataset: The original training dataset
        config: Configuration dictionary
        num_samples: Number of samples to save per class
    """
    print("\n--- Saving Original Training Sample Images ---")
    
    # Create a directory for original samples
    samples_dir = os.path.join(config['results_dir'], "original_samples")
    os.makedirs(samples_dir, exist_ok=True)
    
    # Define inverse normalization for visualization
    inv_normalize = transforms.Normalize(
        mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
        std=[1/0.229, 1/0.224, 1/0.225]
    )
    
    # Create a dictionary to store indices for each class
    class_indices = {i: [] for i in range(config['num_classes'])}
    
    # Collect indices for each class
    for i in range(len(dataset)):
        _, label = dataset[i]
        for class_idx in range(config['num_classes']):
            if label[class_idx] == 1:
                class_indices[class_idx].append(i)
    
    # For each class, save a few samples
    for class_idx, class_name in enumerate(config['classes']):
        indices = class_indices[class_idx]
        
        if len(indices) == 0:
            print(f"No original samples found for class {class_name}")
            continue
            
        # Take a random subset if we have more than num_samples
        if len(indices) > num_samples:
            selected_indices = np.random.choice(indices, num_samples, replace=False)
        else:
            selected_indices = indices
            
        # Get the selected images
        selected_images = []
        for idx in selected_indices:
            img, _ = dataset[idx]
            selected_images.append(img)
        
        selected_images = torch.stack(selected_images)
        
        # Denormalize for visualization
        denormalized_images = inv_normalize(selected_images)
        
        # Save individual images
        for i, img_idx in enumerate(selected_indices):
            img_path = os.path.join(samples_dir, f"{class_name}_original_{i+1}.png")
            vutils.save_image(denormalized_images[i], img_path)
        
        # Save a grid of all samples for this class
        grid_path = os.path.join(samples_dir, f"{class_name}_original_grid.png")
        grid = vutils.make_grid(denormalized_images, nrow=min(num_samples, len(selected_indices)), padding=2, normalize=True)
        vutils.save_image(grid, grid_path)
    
    print(f"Original samples saved to {samples_dir}")


In [ ]:
# --- Main Experiment Loop ---
# Set this to False if GAN models already exist
TRAIN_GANS = True
if TRAIN_GANS:
    for idx, name in enumerate(CONFIG['classes']):
        train_gan_for_class(
            class_name=name,
            class_idx=idx,
            data_dir=CONFIG['data_dir'],
            image_set=CONFIG['image_set_train'],
            gan_transform=gan_transform,
            config=CONFIG
        )
    
    # Save sample images from each class's GAN
    save_gan_samples(CONFIG, num_samples=5)
else:
    print("Skipping GAN training as TRAIN_GANS is set to False.")

# 2. Train and Evaluate Classifier with different augmentation levels
results_map = {}

for n_samples in CONFIG['augmentation_samples']:
    print(f"\n===== Experiment: {n_samples} Synthetic Samples Per Class =====")

    # --- Prepare Augmented Dataset ---
    if n_samples > 0:
        synthetic_images, synthetic_labels = generate_synthetic_data(n_samples, CONFIG)
        if synthetic_images is None or synthetic_labels is None:
             print(f"Skipping {n_samples} samples experiment due to generation issues.")
             # results_map[n_samples] = float('nan') # Record failure
             gc.collect()
             torch.cuda.empty_cache()
             continue # Skip to next sample count
        
        # Save synthetic samples for visualization
        save_synthetic_samples(synthetic_images, synthetic_labels, CONFIG, num_samples=5)

        # Use TensorDataset for synthetic data for easy combination
        synthetic_dataset = torch.utils.data.TensorDataset(synthetic_images, synthetic_labels)
        # Combine original and synthetic datasets with proper weighting
        # Start with lower synthetic weight and gradually increase with more samples
        synthetic_weight = min(0.7, 0.3 + (n_samples / 1000))  # Cap at 0.7 max weight
        augmented_dataset = AugmentedDataset(
            original_dataset=original_train_dataset_classifier, 
            synthetic_images=synthetic_images, 
            synthetic_labels=synthetic_labels,
            synthetic_weight=synthetic_weight
        )
        train_loader = DataLoader(
            augmented_dataset, 
            batch_size=CONFIG['classifier_batch_size'], 
            shuffle=True, 
            num_workers=2, 
            pin_memory=True
        )
        print(f"Augmented training set size: {len(augmented_dataset)}")
        # Clean up large tensors
        del synthetic_images, synthetic_labels, synthetic_dataset
    else:
        # Baseline (no augmentation)
        print(f"Using original training set size: {len(original_train_dataset_classifier)}")
        train_loader = DataLoader(original_train_dataset_classifier, batch_size=CONFIG['classifier_batch_size'], shuffle=True, num_workers=2, pin_memory=True)

    # --- Train Classifier ---
    # Initialize a new classifier for each experiment to ensure fair comparison
    classifier = get_classifier(CONFIG['num_classes'], pretrained=True)
    trained_classifier, best_val_map_during_train = train_classifier(classifier, train_loader, val_loader, CONFIG)

    # --- Evaluate Final Model on Validation Set ---
    # We could re-evaluate the best loaded model, or just use the best mAP recorded during training
    # final_map, _ = evaluate_classifier(trained_classifier, val_loader, nn.BCEWithLogitsLoss(), CONFIG)
    final_map = best_val_map_during_train # Use the best mAP achieved during training epochs
    print(f"--- Final mAP for {n_samples} samples/class: {final_map:.4f} ---")
    results_map[n_samples] = final_map

    # --- Clean up GPU memory before next iteration ---
    del classifier, trained_classifier, train_loader
    if 'augmented_dataset' in locals(): del augmented_dataset # Delete if it exists
    gc.collect()
    torch.cuda.empty_cache()
    print("-" * 50)


In [3]:

# --- 4. Plotting and Discussion ---

print("\n===== Final Results =====")
sample_counts = sorted(results_map.keys())
map_scores = [results_map[s] for s in sample_counts]

for s, m in zip(sample_counts, map_scores):
    print(f"Samples per class: {s}, mAP: {m:.4f}")

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(sample_counts, map_scores, marker='o')
plt.title('Classifier mAP vs. Number of Synthetic Samples per Class')
plt.xlabel('Number of Synthetic Samples per Class Added')
plt.ylabel('Mean Average Precision (mAP) on Validation Set')
plt.xticks(sample_counts) # Ensure x-axis ticks match the tested values
plt.grid(True)
plt.ylim(bottom=max(0, min(map_scores) - 0.05), top=max(map_scores) + 0.05) # Adjust y-axis limits
results_plot_path = os.path.join(CONFIG['results_dir'], "map_vs_augmentation.png")
plt.savefig(results_plot_path)
print(f"Results plot saved to {results_plot_path}")
# plt.show() # Avoid showing in Kaggle script mode
plt.close()

# Discussion
print("\n===== Discussion =====")
baseline_map = results_map[0]
print(f"Baseline mAP (0 synthetic samples): {baseline_map:.4f}")

improvement = False
best_map = baseline_map
best_samples = 0
for s in sample_counts:
    if s > 0 and results_map[s] > best_map:
        improvement = True
        best_map = results_map[s]
        best_samples = s

if improvement:
    print(f"Performance IMPROVED with augmentation.")
    print(f"Best mAP: {best_map:.4f} achieved with {best_samples} synthetic samples per class.")
    print("\nPossible reasons for improvement:")
    print("1. Increased Data Diversity: Even low-quality GAN samples might introduce variations not present in the original limited training data, helping the classifier generalize better.")
    print("2. Regularization Effect: Adding noisy or slightly different synthetic data can act as a form of regularization, preventing the classifier from overfitting to the original training samples.")
    print("3. Balancing Classes (Implicitly): While we added the same number per class, if some classes had very few original samples, the relative increase from synthetic data might be larger, potentially helping the classifier learn those classes better (although mAP averages across all).")
else:
    print(f"Performance DID NOT IMPROVE (or worsened) with augmentation.")
    print(f"Best mAP among augmented runs was {best_map:.4f} with {best_samples} samples (compare to baseline {baseline_map:.4f}).")
    print("\nPossible reasons for lack of improvement or worsening:")
    print("1. Poor GAN Quality: The DCGAN might have generated unrealistic or low-quality images (mode collapse, artifacts) that confused the classifier or didn't represent the true data distribution.")
    print("2. Distribution Shift: The synthetic data distribution might be significantly different from the real data distribution, leading the classifier astray.")
    print("3. Simplistic Labeling: Assigning only a single label to synthetic images might contradict the multi-label nature of the real data, potentially harming performance on complex images.")
    print("4. Sufficient Original Data: The original training set might already be large or diverse enough for the chosen classifier architecture, making augmentation less impactful or even detrimental if the synthetic data is noisy.")
    print("5. Suboptimal Augmentation Amount: The tested amounts (100, 200, 500) might not be optimal. Too few samples might not help, while too many poor-quality samples could hurt.")

print("\n--- Experiment Complete ---")


VOC dataset directory not found: ./VOCdevkit/VOC2008
Found existing VOC dataset file: ./VOCtrainval_14-Jul-2008.tar
Extracting dataset...
Extraction complete.
VOC dataset has been downloaded and extracted.
Using device: cuda
Number of classes: 20
Class names: ['aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor']
Loaded 2111 labels for class 'aeroplane'
Loaded 2111 labels for class 'bicycle'
Loaded 2111 labels for class 'bird'
Loaded 2111 labels for class 'boat'
Loaded 2111 labels for class 'bottle'
Loaded 2111 labels for class 'bus'
Loaded 2111 labels for class 'car'
Loaded 2111 labels for class 'cat'
Loaded 2111 labels for class 'chair'
Loaded 2111 labels for class 'cow'
Loaded 2111 labels for class 'diningtable'
Loaded 2111 labels for class 'dog'
Loaded 2111 labels for class 'horse'
Loaded 2111 labels for class 'motorbike'
Loaded 2111 labels fo

/tmp/ipykernel_20/1730140964.py:1007: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  generator.load_state_dict(torch.load(gan_model_path, map_location=config['device']))


Generating samples for class: bicycle
Generating samples for class: bird
Generating samples for class: boat
Generating samples for class: bottle
Generating samples for class: bus
Generating samples for class: car
Generating samples for class: cat
Generating samples for class: chair
Generating samples for class: cow
Generating samples for class: diningtable
Generating samples for class: dog
Generating samples for class: horse
Generating samples for class: motorbike
Generating samples for class: person
Generating samples for class: pottedplant
Generating samples for class: sheep
Generating samples for class: sofa
Generating samples for class: train
Generating samples for class: tvmonitor
GAN samples saved to ./results/gan_samples

===== Experiment: 0 Synthetic Samples Per Class =====
Using original training set size: 2111


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 204MB/s]


--- Starting Classifier Training ---


...

/tmp/ipykernel_20/1730140964.py:849: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=torch.cuda.is_available()):


--- Evaluating Classifier ---


Evaluation:   0%|          | 0/70 [00:00<?, ?it/s]

/tmp/ipykernel_20/1730140964.py:949: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=torch.cuda.is_available()):


Epoch 25/25 - Train Loss: 0.0135, Val Loss: 0.1890, Val mAP: 0.1937, Train Time: 20.35s
EarlyStopping counter: 2 out of 5
Finished Classifier Training.
Loaded best model state with mAP: 0.1973
--- Final mAP for 500 samples/class: 0.1973 ---
--------------------------------------------------

===== Final Results =====
Samples per class: 0, mAP: 0.6132
Samples per class: 100, mAP: 0.4993
Samples per class: 200, mAP: 0.3697
Samples per class: 500, mAP: 0.1973
Results plot saved to ./results/map_vs_augmentation.png

===== Discussion =====
Baseline mAP (0 synthetic samples): 0.6132
Performance DID NOT IMPROVE (or worsened) with augmentation.
Best mAP among augmented runs was 0.6132 with 0 samples (compare to baseline 0.6132).

Possible reasons for lack of improvement or worsening:
1. Poor GAN Quality: The DCGAN might have generated unrealistic or low-quality images (mode collapse, artifacts) that confused the classifier or didn't represent the true data distribution.
2. Distribution Shift: 